# Summary of GMM, SVGMM, DeepG, and DeepSVG for Unsupervised Image Segmentation

## GMM Algorithm and Drawbacks

The Gaussian Mixture Model (GMM) is used for unsupervised image segmentation, assuming pixel intensities follow a mixture of Gaussian distributions for different classes.

Key components:
- Pixel density: $ p(I(x) | s(x) = k) = g(I(x) | \mu_k, \Sigma_k) $, a multivariate Gaussian.
- Mixture density: $ p(I(x)) = \sum_k \pi_k g(I(x) | \mu_k, \Sigma_k) $.
- Joint density (pixel independence): $ p(I) = \prod_x \sum_k \pi_k g(I(x) | \mu_k, \Sigma_k) $.
- Minimize NLL: $ \text{NLL} = -\frac{1}{|\Omega|} \sum_x \log(\sum_k \pi_k g(I(x) | \mu_k, \Sigma_k)) $.

EM Algorithm:
- Initialize parameters.
- E-step: $ w_{xk} = \frac{\pi_k g(I(x) | \mu_k, \Sigma_k)}{\sum_{k'} \pi_{k'} g(I(x) | \mu_{k'}, \Sigma_{k'})} $.
- M-step: Update $\pi_k, \mu_k, \Sigma_k$ using $w_{xk}$.
- Label by max $w_{xk}$.

**Drawbacks**:
- Ignores spatial pixel correlations (independence assumption).
- Global parameters limit flexibility.
- Computationally slow EM iterations.
- Non-optimal labeling.
- Sensitive to initialization and local maxima.

## SVGMM and DeepG: Extensions and Resolved Issues

**SVGMM (Spatially Variant GMM)**:
- Uses pixel-specific mixing weights $\Pi_k(x)$.
- NLL: $ \text{NLL}_V = -\frac{1}{|\Omega|} \sum_x \log(\sum_k \Pi_k(x) g(I(x) | \mu_k, \Sigma_k)) $.
- EM similar, but updates $\Pi_k(x) = w_{xk}$; labels converge to binary.

**Resolved**: Improves labeling flexibility (pixel-specific weights), but still lacks spatial regularization.

**DeepG**:
- Integrates CNN (U-Net) to predict weights $w = \phi_\theta(I)$.
- Replaces E-step with gradient descent on $\theta$ to minimize NLL.
- Iterates gradient-step and M-step.

**Resolved**: Adds spatial awareness via CNN convolutions (deep image prior), smoothing segmentations; faster post-training.

## DeepSVG Algorithm and Advantages

**DeepSVG**:
- Combines SVGMM with CNN for spatially variant weights.
- Algorithm:
  - Initialize $\theta$, predict $w = \phi_\theta(I)$.
  - M-step for $\Pi, \mu, \Sigma$.
  - Iterate: Gradient-step $\theta \leftarrow \theta - \alpha \nabla_\theta \text{NLL}_V$, update $w$; M-step.
- Optional regularization: Add $\lambda r$ (e.g., mean prior).
- Multi-image training for generalization.

**Advantages**:
- Over SVGMM: Incorporates spatial regularization via CNN.
- Over DeepG: Spatially variant weights for better flexibility and binary label convergence.
- Overall: Smoother/more accurate results (e.g., higher Dice scores), faster inference, easy regularization, good generalization.

## Reason for Gradient-Based E-Step in DeepSVG

Traditional E-step computes $w_{xk}$ per pixel independently, ignoring neighbors. Replacement with gradient descent on CNN parameters allows holistic prediction from the full image, incorporating spatial correlations via convolutions. This adds implicit regularization, overcomes independence, enables faster/regularized optimization, and improves accuracy/flexibility.

# Deep Learning and GMM
Chose one of the options below and report the results.

1. DeepSVG algorithm in the paper "Deep Gaussian mixture model for unsupervised image segmentation".
2. A suggested algorithm utilizing deep neural networks and GMM.

First option is chosen.

In [11]:
!git clone https://github.com/matthi99/deepGMM.git
!git clone https://github.com/Ehsanacc/iMIAP.git

fatal: destination path 'deepGMM' already exists and is not an empty directory.
fatal: destination path 'iMIAP' already exists and is not an empty directory.


In [12]:
import os
import zipfile
import tarfile
import shutil

# Define paths
data_folder = '/kaggle/working/iMIAP/Assignment 2/DATA'  # Path to DATA folder
output_dir = '/kaggle/working/iMIAP/Assignment 2/DATA'       # Parent directory for extracted files

# Ensure DATA folder exists
if not os.path.exists(data_folder):
    print(f"Error: {data_folder} does not exist.")
else:
    # Iterate through files in DATA folder
    for file_name in os.listdir(data_folder):
        file_path = os.path.join(data_folder, file_name)
        
        # Check if the file is an archive
        if os.path.isfile(file_path):
            if file_name.endswith('.zip'):
                print(f"Extracting {file_name}...")
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(output_dir)
                print(f"Extracted {file_name} to {output_dir}")
            
            elif file_name.endswith(('.tar', '.tar.gz', '.tgz')):
                # print(f"Extracting {file_name}...")
                with tarfile.open(file_path, 'r:*') as tar_ref:
                    tar_ref.extractall(output_dir)
                # print(f"Extracted {file_name} to {output_dir}")
            
            else:
                print(f"Skipping {file_name}: Unsupported archive format")
        
        else:
            print(f"Skipping {file_name}: Not a file")

    print("Extraction complete.")

Extracting MyoPS 2020 Dataset.zip...
Extracted MyoPS 2020 Dataset.zip to /kaggle/working/iMIAP/Assignment 2/DATA
Skipping preprocessed: Not a file
Skipping sigma_data.npy: Unsupported archive format
Skipping pi_data.npy: Unsupported archive format
Skipping mu_data.npy: Unsupported archive format
Skipping MyoPS 2020 Dataset: Not a file
Extraction complete.


In [13]:
import os
import zipfile
import tarfile
import shutil

# Define paths
data_folder = '/kaggle/working/iMIAP/Assignment 2/DATA/MyoPS 2020 Dataset'  # Path to DATA folder
output_dir = '/kaggle/working/iMIAP/Assignment 2/DATA/MyoPS 2020 Dataset'       # Parent directory for extracted files

# Ensure DATA folder exists
if not os.path.exists(data_folder):
    print(f"Error: {data_folder} does not exist.")
else:
    # Iterate through files in DATA folder
    for file_name in os.listdir(data_folder):
        file_path = os.path.join(data_folder, file_name)
        
        # Check if the file is an archive
        if os.path.isfile(file_path):
            if file_name.endswith('.zip'):
                print(f"Extracting {file_name}...")
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(output_dir)
                print(f"Extracted {file_name} to {output_dir}")
            
            elif file_name.endswith(('.tar', '.tar.gz', '.tgz')):
                # print(f"Extracting {file_name}...")
                with tarfile.open(file_path, 'r:*') as tar_ref:
                    tar_ref.extractall(output_dir)
                # print(f"Extracted {file_name} to {output_dir}")
            
            else:
                print(f"Skipping {file_name}: Unsupported archive format")
        
        else:
            print(f"Skipping {file_name}: Not a file")

    print("Extraction complete.")

Extracting train25_myops_gd.zip...
Extracted train25_myops_gd.zip to /kaggle/working/iMIAP/Assignment 2/DATA/MyoPS 2020 Dataset
Skipping train25: Not a file
Extracting train25.zip...
Extracted train25.zip to /kaggle/working/iMIAP/Assignment 2/DATA/MyoPS 2020 Dataset
Skipping train25_myops_gd: Not a file
Extraction complete.


In [14]:
import os

# Define the file path
file_path = "/kaggle/working/deepGMM/preprocessing.py"

# Check if the file exists
if not os.path.exists(file_path):
    print(f"Error: {file_path} does not exist.")
else:
    try:
        # Read the file
        with open(file_path, 'r') as file:
            lines = file.readlines()
        
        # Check if the file has enough lines
        if len(lines) < 90:
            print(f"Error: {file_path} has only {len(lines)} lines, cannot modify line 90.")
        else:
            # Modify line 15 (index 14, 0-based)
            lines[14] = 'data_folder = "/kaggle/working/iMIAP/Assignment 2/DATA"\n'

            # lines[28] = 'print(f"len(files_C0): {len(files_C0)}, len(files_T2): {len(files_T2)}, len(files_gt): {len(files_T2)}, len(files_LGE): {len(files_LGE)}")\n'
            # lines[32] = '    print(f"im_C0: {im_C0}")\n'

            # lines[72] = '    print(f"data[\\\'LGE\\\'].shape[0]: {data[\'LGE\'].shape[0]}")\n'
            # lines[73] = '    print(f"len(data[\\\'C0\\\']): {len(data[\'C0\'])}, len(data[\\\'LGE\\\']): {len(data[\'LGE\'])}, len(data[\\\'T2\\\']): {len(data[\'T2\'])}, len(data[\\\'masks\\\']): {len(data[\'masks\'])}")\n'
            # Modify line 90 (index 89, 0-based)
            lines[75] = '    for i in range(min(len(data[\'C0\']), len(data[\'T2\']), len(data[\'LGE\']), len(data[\'masks\']))):\n'
            lines[89] = 'folder = "/kaggle/working/iMIAP/Assignment 2/DATA/preprocessed/myops_2d/"\n'
            
            # Write the modified content back to the file
            with open(file_path, 'w') as file:
                file.writelines(lines)
            
            print(f"Successfully modified {file_path} at lines 15 and 90.")
            
    except IOError as e:
        print(f"Error: Failed to read or write {file_path}. {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

Successfully modified /kaggle/working/deepGMM/preprocessing.py at lines 15 and 90.


In [15]:
!python /kaggle/working/deepGMM/preprocessing.py

Data prepared!


In [16]:
import os

# Define the file path
file_path = "/kaggle/working/deepGMM/deepG_train.py"

# Check if the file exists
if not os.path.exists(file_path):
    print(f"Error: {file_path} does not exist.")
else:
    try:
        # Read the file
        with open(file_path, 'r') as file:
            lines = file.readlines()
        
        # Check if the file has enough lines
        if len(lines) < 90:
            print(f"Error: {file_path} has only {len(lines)} lines, cannot modify line 90.")
        else:
            # Modify line 15 (index 14, 0-based)
            lines[71] = 'mu_data = torch.from_numpy(np.load("/kaggle/working/iMIAP/Assignment 2/DATA/mu_data.npy").astype("float32")).to(device)\n'

            lines[80] = 'cd_train = CardioDataset(folder="/kaggle/working/iMIAP/Assignment 2/DATA/preprocessed/myops_2d/" ,z_dim=False,  **setup_train)\n'
            # Write the modified content back to the file
            with open(file_path, 'w') as file:
                file.writelines(lines)
            
            print(f"Successfully modified {file_path} at lines 15 and 90.")
            
    except IOError as e:
        print(f"Error: Failed to read or write {file_path}. {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

Successfully modified /kaggle/working/deepGMM/deepG_train.py at lines 15 and 90.


In [17]:
!python /kaggle/working/deepGMM/deepG_train.py --type "deepSVG"
!python /kaggle/working/deepGMM/deepG_train.py --type "deepSVG" --tol XXX --max_epochs 200

usage: deepG_train.py [-h] [--max_epochs MAX_EPOCHS] [--min_epochs MIN_EPOCHS]
                      [--batchsize BATCHSIZE] [--type TYPE] [--lam LAM] [--tol TOL]
deepG_train.py: error: argument --tol: invalid float value: 'XXX'


In [18]:
import os

# Define the file path
file_path = "/kaggle/working/deepGMM/deepG_pred.py"

# Check if the file exists
if not os.path.exists(file_path):
    print(f"Error: {file_path} does not exist.")
else:
    try:
        # Read the file
        with open(file_path, 'r') as file:
            lines = file.readlines()

        # Check if the file has enough lines
        if len(lines) < 71:
            print(f"Error: {file_path} has only {len(lines)} lines, cannot modify line 90.")
        else:
            lines[28] = '    files.append([os.path.join("/kaggle/working/iMIAP/Assignment 2/DATA/preprocessed/myops_2d/",f) for f in os.listdir("/kaggle/working/iMIAP/Assignment 2/DATA/preprocessed/myops_2d/") if f.startswith(patient)])\n'
            with open(file_path, 'w') as file:
                file.writelines(lines)

            print(f"Successfully modified {file_path} at lines 15 and 90.")

    except IOError as e:
        print(f"Error: Failed to read or write {file_path}. {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

Successfully modified /kaggle/working/deepGMM/deepG_pred.py at lines 15 and 90.


In [19]:
!python /kaggle/working/deepGMM/deepG_pred.py --type "deepSVG"

Results saved in RESULTS_FOLDER/deepSVG/multiple_images/lam=1/


Results are available in the zip file named deegSVG Results.zip

In [20]:
import zipfile
import os

def zip_folder(folder_path, output_path):
    # Create a zip file
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Walk through the folder
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                # Create the full file path
                file_path = os.path.join(root, file)
                # Calculate the relative path for the file in the zip
                rel_path = os.path.relpath(file_path, os.path.dirname(folder_path))
                # Add file to zip
                zipf.write(file_path, rel_path)

# Define paths
folder_to_zip = '/kaggle/working/RESULTS_FOLDER/deepSVG/multiple_images/lam=5.0'
output_zip = '/kaggle/working/lam_5.0_results.zip'

# Create the zip file
zip_folder(folder_to_zip, output_zip)
print(f"Zip file created at: {output_zip}")

Zip file created at: /kaggle/working/lam_5.0_results.zip
